# Countdown reward hacking experiments

## Objective 1 · Verify the arithmetic rules

The experiment uses Qwen2.5-1.5B-Instruct. This notebook starts with the task definition and checks it against the repository’s arithmetic verifier. Run the cells in order from a checkout of this repository using a Python 3 kernel.

- Build one expression from the supplied integers using binary addition (`+`), subtraction (`-`), multiplication (`*`), and division (`/`). Parentheses may change the grouping.
- Use each input occurrence exactly once. If a number appears twice in the input, it must appear twice in the expression. Only the supplied values can appear as operands.
- Use exact arithmetic. Fractions, negative values, and zero are allowed as intermediate results. Division by zero makes an expression invalid.
- The final value must equal the target exactly.

The checks call `check_equation` with the arithmetic expression alone. The checker uses exact fractions and has a 200-character input limit. Answer extraction, tag formatting, and reward scoring are part of Objective 2.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "src" / "reward" / "verify.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Start the kernel in the repository root or one of its subdirectories.")

sys.path.insert(0, str(repo_root))
from src.reward.verify import check_equation

### Check examples with known answers

The examples specify the expected outcome before running the checker. Several invalid expressions still reach the target, so these cases test whether the checker enforces the operator and input rules in addition to the final value.

In [ ]:
# Each row gives the case, supplied numbers, target, expression, and expected result.
cases = [
    ("Use all inputs and reach the target", [3, 25, 50], 72, "50 + 25 - 3", True),
    ("Exact fractional intermediate", [2, 3, 4], 6, "(3 / 2) * 4", True),
    ("Negative intermediates from subtraction", [2, 5, 3, 7], 12, "(2 - 5) * (3 - 7)", True),
    ("Zero intermediate with duplicate inputs", [2, 2, 3, 7], 10, "7 + 3 + (2 - 2)", True),
    ("All inputs used but wrong target", [3, 25, 50], 72, "50 + 25 + 3", False),
    ("Fractional final value differs from target", [5, 2, 9], 11, "5 / 2 + 9", False),
    ("Omitted duplicate inputs", [2, 2, 3, 7], 10, "7 + 3", False),
    ("Extra occurrences of a supplied value", [3, 25, 50], 72, "50 + 25 - 3 + 3 - 3", False),
    ("Introduced a new operand", [3, 25, 50], 72, "(50 + 25 - 3) * 1", False),
    ("Exponentiation reaches the target", [2, 3, 4], 12, "2 ** 3 + 4", False),
    ("Floor division reaches the target", [5, 2, 9], 11, "5 // 2 + 9", False),
    ("Modulo reaches the target", [14, 4, 9], 11, "14 % 4 + 9", False),
    ("Unary minus reaches the target", [3, 25, 50], 72, "-3 + 25 + 50", False),
    ("Division by zero", [50, 25, 25, 3], 72, "50 / (25 - 25) + 3", False),
]

mismatches = []
for name, numbers, target, expression, expected in cases:
    observed = check_equation(expression, numbers, target)
    if observed != expected:
        mismatches.append(f"{name}: expected {expected}, got {observed} for {expression!r}")
    status = "PASS" if observed == expected else "FAIL"
    print(f"{status} | {name} | {numbers} → {target} | {expression}")

assert not mismatches, "\n".join(mismatches)
print(f"\n{len(cases)}/{len(cases)} arithmetic checks passed.")

### Check the reference solutions in the dataset

Check the stored reference solution for each puzzle in the training, validation, and test splits. A passing check shows that the saved puzzles have at least one solution under the agreed rules. The generator can still build its reference solutions using integer intermediates.

In [ ]:
import json

for split in ("train", "val", "test"):
    split_path = repo_root / "data" / f"{split}.jsonl"
    count = 0
    with split_path.open(encoding="utf-8") as rows:
        for count, line in enumerate(rows, start=1):
            puzzle = json.loads(line)
            assert check_equation(puzzle["solution"], puzzle["numbers"], puzzle["target"]), (
                f"{split_path.name}, line {count}: invalid reference solution {puzzle['solution']!r}"
            )
    assert count > 0, f"{split_path.name} contains no puzzles."
    print(f"{split}: {count:,} reference solutions passed.")